# QANTA 2021: Accuracy-Fixed Follow-Up and 50-Question Pilot (v3)

This notebook resolves the four accuracy bottlenecks identified in the initial Stage 3 follow-up run (`0/10` accuracy):
1. **100% Target Topic Coverage in the 500-Article Corpus:** `scripts/build_wikipedia_corpus.py` now prioritizes the 50 pilot question Wikipedia topics (with canonical title resolution for malformed QANTA page strings like `ⅆ -> Duel` and `THE_Tunnel -> The Tunnel (novel)`) alongside 450 seeded QANTA training distractors, raising target coverage from `0/50` to `50/50` while preserving `{{lang|...}}` titles and prepending article titles to every chunk.
2. **Title-Aware, Stopword-Filtered BM25 Retrieval:** `BM25` indexes and returns Wikipedia article `title` fields (with a 2x title weight boost) and filters quizbowl boilerplate stopwords (`for 10 points`, `ftp`, `name this`, etc.).
3. **Anti-Clue-Copying Prompt (`qa-answer-v3`) & System Prompt:** Explicitly instructs the model to identify the unnamed target entity (`this painting`, `this novel`, `this piece`, `these constructs`) and never echo an author, artist, composer, character, or example already named in the clues.
4. **Base Model Upgrade (`Qwen/Qwen2.5-3B-Instruct`):** Steps up from the 0.5B model (~1.1 GB VRAM) to `Qwen/Qwen2.5-3B-Instruct` (~6.2 GB VRAM on a 15 GB Colab Tesla T4 GPU), while allowing you to switch back to `Qwen/Qwen2.5-0.5B-Instruct` if desired.

## 1. Persistent workspace

Stage 3 has its own Drive directory. The earlier project, its original results, and its caches remain available for comparison. The two project paths identify the existing Stage 2 inputs and the new Stage 3 workspace.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
STAGE2 = Path('/content/drive/MyDrive/AdaptiveQA2021_consistency/adaptive-confidence-agent')
PROJECT = Path('/content/drive/MyDrive/AdaptiveQA2021_consistency/adaptive-qa-stage03-v3')
assert STAGE2 != PROJECT
print('Stage 2:', STAGE2)
print('Stage 3 (v3):', PROJECT)

## 2. Source provenance

The source archive is exported from a Git commit. Its commit identifier, SHA-256 digest, and file hashes are retained with the experiment. An existing workspace is reused only when its recorded source files still match. The archived code contains no generated results or private correspondence.


In [ ]:
import hashlib, json, re, stat
from io import BytesIO
from pathlib import PurePosixPath
from zipfile import ZipFile
from google.colab import files
receipt_path = PROJECT/'results/source_archive.json'
if receipt_path.exists():
    receipt = json.loads(receipt_path.read_text())
    changed = [name for name, sha in receipt['files'].items()
               if not (PROJECT/name).is_file() or hashlib.sha256((PROJECT/name).read_bytes()).hexdigest() != sha]
    if changed:
        raise RuntimeError(f'Source differs from the archived version: {changed}. Preserve this run; use a new project directory for changed code.')
else:
    if PROJECT.exists() and any(PROJECT.iterdir()):
        raise RuntimeError('This Stage 3 folder already contains files without an archive receipt. Use a new empty project directory.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload one stage3-colab-source.zip exported by Git.')
    archive_name, archive_bytes = next(iter(uploaded.items()))
    source_files = {}
    with ZipFile(BytesIO(archive_bytes)) as archive:
        commit = archive.comment.decode('ascii').strip()
        if not re.fullmatch(r'[0-9a-f]{40,64}', commit):
            raise ValueError('The ZIP has no Git archive commit identifier. Use stage3-colab-source.zip from the private Git guide.')
        for member in archive.infolist():
            parts = PurePosixPath(member.filename).parts
            if not parts or parts[0] != 'adaptive-qa' or '..' in parts or '\\' in member.filename:
                raise ValueError('Unexpected archive path: '+member.filename)
            if stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError('Source archive must not contain symlinks')
            if member.is_dir():
                continue
            relative = PurePosixPath(*parts[1:]).as_posix()
            if relative in source_files:
                raise ValueError('Duplicate archive entry')
            source_files[relative] = archive.read(member)
    for required in ['src/adaptive_qa/pilot.py', 'configs/stage03_followup10.yaml', 'configs/stage03_pilot50.yaml']:
        if required not in source_files:
            raise ValueError('Stage 3 source missing: '+required)
    PROJECT.mkdir(parents=True, exist_ok=True)
    for relative, data in source_files.items():
        target=PROJECT/relative; target.parent.mkdir(parents=True,exist_ok=True); target.write_bytes(data)
    receipt = {'exported_git_commit':commit, 'archive_sha256':hashlib.sha256(archive_bytes).hexdigest(),
               'files':{name:hashlib.sha256(data).hexdigest() for name,data in source_files.items()}}
    receipt_path.parent.mkdir(parents=True,exist_ok=True)
    receipt_path.write_text(json.dumps(receipt,indent=2))
    archive_dir=PROJECT.parent/'stage03_source_archives'; archive_dir.mkdir(exist_ok=True)
    (archive_dir/(commit+'-'+receipt['archive_sha256'][:12]+'.zip')).write_bytes(archive_bytes)
print('Exported Git commit:',receipt['exported_git_commit'])


## 3. Runtime and execution records

Each script runs in a separate Python process. Its console output, exit status, and elapsed time are retained even if the script fails. Model-call ledgers separately distinguish cached observations, successful calls, failed attempts, and unfinished attempts. The Transformers version matches the recorded Stage 2 run; other installed versions are saved with the run.


In [ ]:
import subprocess, sys, time
from datetime import datetime, timezone

def run_script(script, *args):
    folder=PROJECT/'results/logs'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    folder.mkdir(parents=True)
    command=[sys.executable,'-u',script,*map(str,args)]
    started=time.perf_counter()
    with (folder/'stdout.txt').open('w',encoding='utf-8') as log:
        process=subprocess.Popen(command,cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');log.write(line);log.flush()
            returncode=process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
        finally:
            (folder/'command.json').write_text(json.dumps({'command':command,'returncode':process.poll(),
                'wall_seconds':time.perf_counter()-started},indent=2))
    if returncode:
        raise RuntimeError(f'Script failed; records are in {folder}')
    return folder

subprocess.run([sys.executable,'-m','pip','install','-e','.[gpu]','transformers==4.57.6','mwparserfromhell'],cwd=PROJECT,check=True)
installed=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
(PROJECT/'results/runtime_packages.txt').write_text(installed)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No free Colab GPU is active. Preserve the workspace and resume when a GPU is available.')
print('GPU:',torch.cuda.get_device_name(0))
print('Device memory (GiB):',torch.cuda.get_device_properties(0).total_memory/2**30)

## 4. Frozen development inputs

The reviewed 10-question subset, the full 50-question development set, their question-level splits, and the evidence corpus are copied from Stage 2 after their hashes are checked. No questions, aliases, article titles, or passages are reselected for this prompt comparison. The split named “test” inside this pilot is still development data.


In [ ]:
run_script('scripts/import_stage2_data.py','--from-project',str(STAGE2))
# Rebuild the 500-article corpus so all 50 pilot topics are covered alongside 450 Stage 2 distractors
run_script('scripts/build_wikipedia_corpus.py','--contact','https://qanta-org.github.io','--overwrite-corpus')
run_script('scripts/import_stage2_data.py','--check-only')
run_script('scripts/run_tests.py')

## 5. Fixed settings and call limits

The follow-up and full pilot share a response cache and identical generation settings. A complete 10-question follow-up uses 100 unique calls: 40 repeated root answers, 40 post-retrieval answers, and 20 independent direct answers. Those observations belong to the 50-question set, so the full collection adds 400 unique calls when the cache is intact. The original Stage 2 calls remain additional work. Failure retries or changed settings can increase actual effort.


In [ ]:
import yaml
CONFIG10='configs/stage03_followup10.yaml'
CONFIG50='configs/stage03_pilot50.yaml'
# Set MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct' here if you want to isolate prompt + corpus fixes on the 0.5B model
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
run_script('scripts/pin_colab_model.py','--configs',CONFIG10,CONFIG50,'--model',MODEL_NAME,'--force')
c10=yaml.safe_load((PROJECT/CONFIG10).read_text())
c50=yaml.safe_load((PROJECT/CONFIG50).read_text())
assert c10['model']==c50['model']
OUT10=PROJECT/'results/runs'/c10['experiment_id']
OUT50=PROJECT/'results/runs'/c50['experiment_id']
run_script('scripts/run.py','estimate','--config',CONFIG10)
run_script('scripts/run.py','estimate','--config',CONFIG50)
print('Model:', c10['model']['name'], '@', c10['model']['revision'])
print('Follow-up:',OUT10)
print('Pilot:',OUT50)

## 6. Revised-prompt baseline on ten questions

This first observation contains 40 generations across 20 prefix states. It isolates answer behavior before retrieval. Parse validity and literal placeholder flags are reviewed separately from correctness; a syntactically valid answer can be wrong. The prompt change remains an untested hypothesis until these real outputs are inspected.


In [ ]:
run_script('scripts/run.py','baseline','--config',CONFIG10,'--execute-model')


In [ ]:
import csv
from IPython.display import display, Markdown
report10=json.loads((OUT10/'pilot_report.json').read_text())
display(Markdown((OUT10/'pilot_report.md').read_text()))
with (OUT10/'tables/sample_review.csv').open(newline='',encoding='utf-8') as f:
    for row in csv.DictReader(f):
        print(row['question_id'],row['requested_fraction'],row['sample_index'],
              'answer=',row['prediction'],'gold=',row['ground_truth'],
              'valid=',row['parse_valid'],'placeholder=',row['placeholder_flag'])
print('Detailed raw outputs:',OUT10/'tables/sample_review.csv')


## 7. Retrieval follow-up on the same questions

The collection adds post-retrieval observations and the independent direct-answer comparison. Existing root samples are read from cache. The verification checks that every configured question and prefix has the expected tree and recorded usage. The retrieval review includes the actual passages returned by BM25.


In [ ]:
run_script('scripts/run.py','collect','--config',CONFIG10,'--execute-model')
run_script('scripts/verify_collection.py','--config',CONFIG10)
run_script('scripts/run.py','pilot-report','--config',CONFIG10)
display(Markdown((OUT10/'pilot_report.md').read_text()))
print('Raw sample review:',OUT10/'tables/sample_review.csv')
print('Retrieved passage review:',OUT10/'tables/retrieval_review.csv')
files.download(str(OUT10/'tables/sample_review.csv'))
files.download(str(OUT10/'tables/retrieval_review.csv'))


## 8. Follow-up interpretation

The expansion decision is recorded after reviewing raw answers and retrieved passages. A low score alone is not a reason to discard observations. Persistent placeholder output, malformed responses, a scoring mismatch, or unusable evidence needs explanation before expansion. The review records limitations without claiming that agreement is calibrated confidence.


In [ ]:
verification=json.loads((OUT10/'collection_verification.json').read_text())
assert verification['status']=='PASSED' and verification['root_states']==20 and verification['nodes']==40
notes=input('Write a brief review of answer format, answer quality and retrieved evidence: ').strip()
if not notes:
    raise ValueError('A written review is needed before the expansion decision is recorded.')
decision=input('Decision after reviewing the follow-up: proceed or pause? ').strip().lower()
if decision not in {'proceed','pause'}:
    raise ValueError('Use proceed or pause.')
review={'decision':decision,'notes':notes,'reviewed_utc':datetime.now(timezone.utc).isoformat(),
        'followup_fingerprint':json.loads((OUT10/'manifest.json').read_text())['fingerprint'],
        'sample_review_sha256':hashlib.sha256((OUT10/'tables/sample_review.csv').read_bytes()).hexdigest(),
        'retrieval_review_sha256':hashlib.sha256((OUT10/'tables/retrieval_review.csv').read_bytes()).hexdigest()}
(PROJECT/'results/stage03_followup_review.json').write_text(json.dumps(review,indent=2))
print('Recorded:',decision)


## 9. Fifty-question development collection

This run reuses the ten-question observations and collects the remaining forty questions. It contains 100 prefix states, 200 observed nodes, and 500 logical generation calls. Each retrieval is limited to three 150-word passages. The original 2,048-input-token and 128-output-token limits remain in force. No controller fitting or additional actions occur here.


In [ ]:
review=json.loads((PROJECT/'results/stage03_followup_review.json').read_text())
if review['decision']!='proceed':
    raise RuntimeError('The follow-up is paused. Preserve the reports and resolve the recorded issue before expanding.')
if review['followup_fingerprint']!=json.loads((OUT10/'manifest.json').read_text())['fingerprint']:
    raise RuntimeError('The follow-up differs from the reviewed run.')
for key, name in [('sample_review_sha256','sample_review.csv'),('retrieval_review_sha256','retrieval_review.csv')]:
    if review[key]!=hashlib.sha256((OUT10/'tables'/name).read_bytes()).hexdigest():
        raise RuntimeError('Follow-up outputs changed since review.')
run_script('scripts/import_stage2_data.py','--check-only')
run_script('scripts/run.py','collect','--config',CONFIG50,'--execute-model')
run_script('scripts/verify_collection.py','--config',CONFIG50)
run_script('scripts/run.py','pilot-report','--config',CONFIG50)
run_script('scripts/summarize_stage3_cost.py')


## 10. Pilot observations and private backup

The report separates direct, root, and post-retrieval correctness; repeated but wrong answers; placeholder output; logical cost; and the actual recorded work across retries. Half-question rows are grouped by the requested prefix fraction, while the actual rounded word fractions are retained. All results are descriptive. A private backup contains the source, frozen inputs, cached outputs, logs, review notes and reports; it belongs outside the public repository.


In [ ]:
display(Markdown((OUT50/'pilot_report.md').read_text()))
print((PROJECT/'results/stage03_cost_summary.json').read_text())


In [ ]:
BACKUPS=PROJECT.parent/'stage03_private_backups'
run_script('scripts/export_stage3_backup.py','--destination',str(BACKUPS))
backup=max(BACKUPS.glob('adaptiveqa-stage03-private-backup-*.zip'),key=lambda p:p.stat().st_mtime)
print('Saved private backup:',backup)
files.download(str(backup))
